In [19]:
import os
import torch
import numpy as np
from collections import defaultdict
import pprint
import pandas as pd

def summarize(values):
    return {
        'mean': np.nanmean(values),
        'sd': np.nanstd(values)
    }

B = 100  # Change as needed
n_i = 4  # Change as needed
result_folder = f'../data/results/B_{B}_n_{n_i}'

stats = {
    'GPmodel': defaultdict(list),
    'GPArealModel': defaultdict(list),
    'VIGP_unlinked': defaultdict(lambda: defaultdict(list))
}

for fname in os.listdir(result_folder):
    if not fname.startswith('results_seed_') or not fname.endswith('.pt'):
        continue
    path = os.path.join(result_folder, fname)
    result = torch.load(path, map_location='cpu')

    for model in ['GPmodel', 'GPArealModel']:
        if model in result:
            m = result[model]
            stats[model]['mu'].append(m.get('beta', [np.nan])[0])
            stats[model]['sigmasq'].append(m.get('sigmasq', np.nan))
            stats[model]['tausq'].append(m.get('tausq', np.nan))
            stats[model]['phi'].append(m.get('phi', np.nan))

    for tau in [0.1, 0.3, 0.5]:
        key = f'VIGP_unlinked_tau_{tau}'
        if key in result:
            vi = result[key]
            mu = vi.get('mu_lambda_beta', np.nan)
            sig_beta = vi.get('sigmasq_lambda_beta', np.nan)

            a1, b1 = vi.get('lambda_a1', np.nan), vi.get('lambda_b1', np.nan)
            a2, b2 = vi.get('lambda_a2', np.nan), vi.get('lambda_b2', np.nan)

            sigmasq = b1 / (a1 - 1) if a1 > 1 else np.nan
            tausq = b2 / (a2 - 1) if a2 > 1 else np.nan

            sd_sigmasq = (b1**2) / ((a1 - 1)**2 * (a1 - 2)) if a1 > 2 else np.nan
            sd_tausq = (b2**2) / ((a2 - 1)**2 * (a2 - 2)) if a2 > 2 else np.nan

            phi = vi.get('mean_phi', np.nan)

            stats['VIGP_unlinked'][tau]['mu'].append(mu)
            stats['VIGP_unlinked'][tau]['sigmasq'].append(sigmasq)
            stats['VIGP_unlinked'][tau]['tausq'].append(tausq)
            stats['VIGP_unlinked'][tau]['phi'].append(phi)
            stats['VIGP_unlinked'][tau]['sd_sigmasq'].append(sd_sigmasq)
            stats['VIGP_unlinked'][tau]['sd_tausq'].append(sd_tausq)
            stats['VIGP_unlinked'][tau]['sd_beta'].append(np.sqrt(sig_beta))

summary = {
    model: {k: summarize(v) for k, v in stats[model].items()}
    for model in ['GPmodel', 'GPArealModel']
}
summary['VIGP_unlinked'] = {
    tau: {
        'beta': {'mean': np.nanmean(v['mu']), 'sd': np.nanmean(v['sd_beta'])},
        'sigmasq': {'mean': np.nanmean(v['sigmasq']), 'sd': np.nanmean(v['sd_sigmasq'])},
        'tausq': {'mean': np.nanmean(v['tausq']), 'sd': np.nanmean(v['sd_tausq'])},
        'phi': {'mean': np.nanmean(v['phi']), 'sd': np.nanstd(v['phi'])}
    } for tau, v in stats['VIGP_unlinked'].items()
}

# Update the summary to use 'beta' instead of 'mu' for GPmodel and GPArealModel
for model in ['GPmodel', 'GPArealModel']:
    summary[model]['beta'] = summary[model].pop('mu')

# Prepare data for the table
data = {
    'GPmodel': [summary['GPmodel']['beta']['mean'], summary['GPmodel']['beta']['sd']],
    'GPArealModel': [summary['GPArealModel']['beta']['mean'], summary['GPArealModel']['beta']['sd']],
}

for tau in [0.1, 0.3, 0.5]:
    data[f'VI_tau_{tau}'] = [
        summary['VIGP_unlinked'][tau]['beta']['mean'],
        summary['VIGP_unlinked'][tau]['beta']['sd']
    ]

# Create a DataFrame for the table
table = pd.DataFrame(data, index=['Mean', 'SD'])

# Add rows for beta, phi, tausq, and sigmasq in "mean (sd)" format
data_extended = {
    'GPmodel': [
        f"{summary['GPmodel']['beta']['mean']:.4f} ({summary['GPmodel']['beta']['sd']:.4f})",
        f"{summary['GPmodel']['phi']['mean']:.4f} ({summary['GPmodel']['phi']['sd']:.4f})",
        f"{summary['GPmodel']['tausq']['mean']:.4f} ({summary['GPmodel']['tausq']['sd']:.4f})",
        f"{summary['GPmodel']['sigmasq']['mean']:.4f} ({summary['GPmodel']['sigmasq']['sd']:.4f})"
    ],
    'GPArealModel': [
        f"{summary['GPArealModel']['beta']['mean']:.4f} ({summary['GPArealModel']['beta']['sd']:.4f})",
        f"{summary['GPArealModel']['phi']['mean']:.4f} ({summary['GPArealModel']['phi']['sd']:.4f})",
        f"{summary['GPArealModel']['tausq']['mean']:.4f} ({summary['GPArealModel']['tausq']['sd']:.4f})",
        f"{summary['GPArealModel']['sigmasq']['mean']:.4f} ({summary['GPArealModel']['sigmasq']['sd']:.4f})"
    ],
}

for tau in [0.1, 0.3, 0.5]:
    data_extended[f'VI_tau_{tau}'] = [
        f"{summary['VIGP_unlinked'][tau]['beta']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['beta']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['phi']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['phi']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['tausq']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['tausq']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['sigmasq']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['sigmasq']['sd']:.4f})"
    ]

# Create a DataFrame for the extended table
table_extended = pd.DataFrame(
    data_extended,
    index=['Beta', 'Phi', 'Tausq', 'Sigmasq']
)

# Display the tables
print(table)
print(table_extended)


       GPmodel  GPArealModel  VI_tau_0.1  VI_tau_0.3  VI_tau_0.5
Mean  7.928946      7.402315    7.722455    7.641274    7.426355
SD    0.139292      0.620306    0.120973    0.098655    0.106004
                 GPmodel     GPArealModel       VI_tau_0.1       VI_tau_0.3  \
Beta     7.9289 (0.1393)  7.4023 (0.6203)  7.7225 (0.1210)  7.6413 (0.0987)   
Phi      5.8339 (1.7048)  5.1924 (2.1687)  3.4914 (0.6235)  3.8376 (0.5351)   
Tausq    0.1893 (0.0710)  0.7299 (0.4229)  4.0502 (0.7664)  3.1106 (0.4953)   
Sigmasq  4.0304 (0.7900)  3.8391 (0.8423)  2.4012 (0.0349)  3.4144 (0.0651)   

              VI_tau_0.5  
Beta     7.4264 (0.1060)  
Phi      3.7997 (0.4837)  
Tausq    2.3445 (0.1567)  
Sigmasq  3.2737 (0.0602)  


/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_5890/1788008002.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  result = torch.load(path, map_location='cpu')

In [20]:
print(f"B = {B}, and n = {n_i}.")
print(table_extended)

B = 100, and n = 4.
                 GPmodel     GPArealModel       VI_tau_0.1       VI_tau_0.3  \
Beta     7.9289 (0.1393)  7.4023 (0.6203)  7.7225 (0.1210)  7.6413 (0.0987)   
Phi      5.8339 (1.7048)  5.1924 (2.1687)  3.4914 (0.6235)  3.8376 (0.5351)   
Tausq    0.1893 (0.0710)  0.7299 (0.4229)  4.0502 (0.7664)  3.1106 (0.4953)   
Sigmasq  4.0304 (0.7900)  3.8391 (0.8423)  2.4012 (0.0349)  3.4144 (0.0651)   

              VI_tau_0.5  
Beta     7.4264 (0.1060)  
Phi      3.7997 (0.4837)  
Tausq    2.3445 (0.1567)  
Sigmasq  3.2737 (0.0602)  


In [23]:
result['VIGP_unlinked_tau_0.1']

{'M_X_star': tensor([[ 1.2576e-03,  1.0035e+00,  8.6913e-05, -1.1250e-03],
         [ 4.2657e-03, -4.8218e-03, -4.8860e-04,  1.0024e+00],
         [ 9.7960e-01, -5.6481e-03,  5.5497e-03,  2.8492e-02],
         [ 2.3399e-02, -2.4999e-03,  9.7703e-01, -8.4892e-04]]),
 'mean_Rphi_inv': tensor([[ 6.2365e+00, -3.2973e-01, -1.9381e+00,  ...,  1.2588e-07,
          -3.4822e-06,  2.0548e-06],
         [-3.2973e-01,  5.5628e+00, -5.0006e-02,  ...,  1.0721e-06,
           1.1032e-06, -3.4696e-06],
         [-1.9381e+00, -5.0006e-02,  9.1084e+00,  ..., -6.5246e-07,
           5.9584e-06, -2.1192e-06],
         ...,
         [ 1.2588e-07,  1.0721e-06, -6.5246e-07,  ...,  4.2235e+00,
          -2.4283e+00,  2.3698e-01],
         [-3.4822e-06,  1.1032e-06,  5.9584e-06,  ..., -2.4283e+00,
           6.1161e+00, -2.7947e+00],
         [ 2.0548e-06, -3.4696e-06, -2.1192e-06,  ...,  2.3698e-01,
          -2.7947e+00,  5.8762e+00]]),
 'V_X_star': tensor([[ 9.6237e-01, -4.0989e-03,  2.8206e-02,  3.2183e-0